<a href="https://colab.research.google.com/github/eversonj1/A-1/blob/main/lumen_community_bank_quality_gates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1 — Project setup

from pathlib import Path
import os
import shutil
import json
import random

PROJECT = Path("/content/lumen-community-bank-quality-gates")

# Start clean if this cell is rerun
if PROJECT.exists():
    shutil.rmtree(PROJECT)

# Create repository structure
folders = [
    "src",
    "tests",
    "contracts",
    "data",
    "data/clean",
    "data/bad",
    "staging",
    "evidence",
    ".github/workflows"
]

for folder in folders:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

# Move into project directory
os.chdir(PROJECT)

print("LUMEN COMMUNITY BANK — QUALITY GATES")
print("=" * 55)
print(f"Project directory: {PROJECT}")
print()
print("Repository structure:")

for folder in folders:
    print(f"✓ {folder}/")

print()
print("✓ Project initialized")
print("✓ Ready to generate synthetic core-banking data")

LUMEN COMMUNITY BANK — QUALITY GATES
Project directory: /content/lumen-community-bank-quality-gates

Repository structure:
✓ src/
✓ tests/
✓ contracts/
✓ data/
✓ data/clean/
✓ data/bad/
✓ staging/
✓ evidence/
✓ .github/workflows/

✓ Project initialized
✓ Ready to generate synthetic core-banking data


In [ ]:
# CELL 2 — Generate clean synthetic core-banking data

import pandas as pd
import numpy as np

np.random.seed(42)
random.seed(42)

NUM_CUSTOMERS = 1800
NUM_ACCOUNTS = 2000
NUM_BRANCHES = 42
NUM_DAYS = 90

END_DATE = pd.Timestamp("2026-10-10")
START_DATE = END_DATE - pd.Timedelta(days=NUM_DAYS - 1)

# --------------------------------------------------
# BRANCH REFERENCE
# --------------------------------------------------

branches = pd.DataFrame({
    "branch_code": [f"B{i:03d}" for i in range(1, NUM_BRANCHES + 1)],
    "branch_name": [f"Lumen Branch {i}" for i in range(1, NUM_BRANCHES + 1)],
    "region": np.random.choice(
        ["North", "South", "East", "West", "Central"],
        NUM_BRANCHES
    )
})

# --------------------------------------------------
# CUSTOMERS
# --------------------------------------------------

customers = pd.DataFrame({
    "customer_id": [f"C{i:06d}" for i in range(1, NUM_CUSTOMERS + 1)],
    "customer_type": np.random.choice(
        ["PERSONAL", "BUSINESS"],
        NUM_CUSTOMERS,
        p=[0.82, 0.18]
    ),
    "state": np.random.choice(
        ["MN", "WI", "IA", "ND", "SD"],
        NUM_CUSTOMERS
    )
})

# --------------------------------------------------
# ACCOUNTS
# --------------------------------------------------

accounts = pd.DataFrame({
    "account_id": [f"A{i:06d}" for i in range(1, NUM_ACCOUNTS + 1)],
    "customer_id": np.random.choice(
        customers["customer_id"],
        NUM_ACCOUNTS
    ),
    "branch_code": np.random.choice(
        branches["branch_code"],
        NUM_ACCOUNTS
    ),
    "account_type": np.random.choice(
        ["CHECKING", "SAVINGS", "MONEY_MARKET"],
        NUM_ACCOUNTS,
        p=[0.55, 0.35, 0.10]
    ),
    "status": np.random.choice(
        ["ACTIVE", "CLOSED", "FROZEN"],
        NUM_ACCOUNTS,
        p=[0.94, 0.04, 0.02]
    ),
    "opened_date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2015-01-01", "2026-06-30"),
            NUM_ACCOUNTS
        )
    )
})

# --------------------------------------------------
# DAILY BALANCES
# --------------------------------------------------

balance_records = []

for _, account in accounts.iterrows():

    starting_balance = round(
        np.random.uniform(250, 25000),
        2
    )

    balance = starting_balance

    for date in pd.date_range(START_DATE, END_DATE):

        # Small realistic daily movement
        movement = np.random.normal(0, 0.025)

        balance = max(
            0,
            balance * (1 + movement)
        )

        balance_records.append({
            "account_id": account["account_id"],
            "balance_date": date,
            "daily_balance": round(balance, 2)
        })

daily_balances = pd.DataFrame(balance_records)

# --------------------------------------------------
# SAVE CLEAN DATA
# --------------------------------------------------

CLEAN = PROJECT / "data" / "clean"

branches.to_csv(
    CLEAN / "branch_reference.csv",
    index=False
)

customers.to_csv(
    CLEAN / "customers.csv",
    index=False
)

accounts.to_csv(
    CLEAN / "accounts.csv",
    index=False
)

daily_balances.to_csv(
    CLEAN / "daily_balances.csv",
    index=False
)

# --------------------------------------------------
# VALIDATION SUMMARY
# --------------------------------------------------

print("SYNTHETIC CORE-BANKING DATA")
print("=" * 60)

print(f"Branches:              {len(branches):,}")
print(f"Customers:             {len(customers):,}")
print(f"Accounts:              {len(accounts):,}")
print(f"Daily balance rows:    {len(daily_balances):,}")
print(f"Balance period:        {START_DATE.date()} to {END_DATE.date()}")

print()
print("CLEAN-DATA CHECKS")
print("=" * 60)

print(
    "Duplicate account IDs:",
    accounts["account_id"].duplicated().sum()
)

print(
    "Duplicate branch codes:",
    branches["branch_code"].duplicated().sum()
)

print(
    "Null customer IDs:",
    accounts["customer_id"].isna().sum()
)

orphan_branches = (
    ~accounts["branch_code"]
    .isin(branches["branch_code"])
).sum()

print(
    "Orphan branch codes:",
    orphan_branches
)

orphan_customers = (
    ~accounts["customer_id"]
    .isin(customers["customer_id"])
).sum()

print(
    "Orphan customer IDs:",
    orphan_customers
)

print()
print("✓ Clean baseline dataset created")
print("✓ Ready for deliberate quality violations")

SYNTHETIC CORE-BANKING DATA
Branches:              42
Customers:             1,800
Accounts:              2,000
Daily balance rows:    180,000
Balance period:        2026-07-13 to 2026-10-10

CLEAN-DATA CHECKS
Duplicate account IDs: 0
Duplicate branch codes: 0
Null customer IDs: 0
Orphan branch codes: 0
Orphan customer IDs: 0

✓ Clean baseline dataset created
✓ Ready for deliberate quality violations


In [ ]:
# CELL 3 — Create deliberately bad dataset

BAD = PROJECT / "data" / "bad"

# Make copies so clean baseline stays untouched
bad_accounts = accounts.copy()
bad_customers = customers.copy()
bad_branches = branches.copy()
bad_balances = daily_balances.copy()

# --------------------------------------------------
# 1. DUPLICATE ACCOUNT ID
# --------------------------------------------------

duplicate_account = bad_accounts.iloc[[0]].copy()
bad_accounts = pd.concat(
    [bad_accounts, duplicate_account],
    ignore_index=True
)

# --------------------------------------------------
# 2. NULL CUSTOMER ID
# --------------------------------------------------

bad_accounts.loc[10, "customer_id"] = None

# --------------------------------------------------
# 3. ORPHAN BRANCH CODE
# --------------------------------------------------

bad_accounts.loc[20, "branch_code"] = "B999"

# --------------------------------------------------
# 4. DUPLICATE BRANCH KEY
#    This recreates the join fan-out incident.
# --------------------------------------------------

duplicate_branch = bad_branches.iloc[[0]].copy()
duplicate_branch["branch_name"] = (
    "Lumen Branch 1 Duplicate"
)

bad_branches = pd.concat(
    [bad_branches, duplicate_branch],
    ignore_index=True
)

# --------------------------------------------------
# 5. BUSINESS-RULE VIOLATION
#    Create >300% day-over-day balance movement.
# --------------------------------------------------

target_account = bad_balances["account_id"].iloc[0]

target_rows = (
    bad_balances[
        bad_balances["account_id"] == target_account
    ]
    .sort_values("balance_date")
    .index
)

# Take one normal day's balance and make the next day
# more than 4x larger, which exceeds a 300% increase.
previous_index = target_rows[39]
spike_index = target_rows[40]

previous_balance = bad_balances.loc[
    previous_index,
    "daily_balance"
]

bad_balances.loc[
    spike_index,
    "daily_balance"
] = round(previous_balance * 5, 2)

# --------------------------------------------------
# SAVE BAD DATA
# --------------------------------------------------

bad_branches.to_csv(
    BAD / "branch_reference.csv",
    index=False
)

bad_customers.to_csv(
    BAD / "customers.csv",
    index=False
)

bad_accounts.to_csv(
    BAD / "accounts.csv",
    index=False
)

bad_balances.to_csv(
    BAD / "daily_balances.csv",
    index=False
)

# --------------------------------------------------
# DEMONSTRATE THE VIOLATIONS
# --------------------------------------------------

duplicate_accounts = (
    bad_accounts["account_id"]
    .duplicated(keep=False)
    .sum()
)

duplicate_branch_codes = (
    bad_branches["branch_code"]
    .duplicated(keep=False)
    .sum()
)

null_customers = (
    bad_accounts["customer_id"]
    .isna()
    .sum()
)

orphan_branches = (
    ~bad_accounts["branch_code"]
    .isin(bad_branches["branch_code"])
).sum()

# Demonstrate fan-out
before_join = len(bad_accounts)

joined = bad_accounts.merge(
    bad_branches,
    on="branch_code",
    how="left"
)

after_join = len(joined)
fanout_rows = after_join - before_join

# Calculate the deliberate balance movement
account_history = (
    bad_balances[
        bad_balances["account_id"] == target_account
    ]
    .sort_values("balance_date")
    .copy()
)

account_history["pct_change"] = (
    account_history["daily_balance"]
    .pct_change()
    .abs()
)

balance_violations = account_history[
    account_history["pct_change"] > 3.0
]

print("DELIBERATE QUALITY VIOLATIONS")
print("=" * 65)

print(f"Bad accounts rows:              {len(bad_accounts):,}")
print(f"Duplicate account rows:         {duplicate_accounts}")
print(f"Null customer IDs:              {null_customers}")
print(f"Orphan branch codes:            {orphan_branches}")
print(f"Duplicate branch-key rows:      {duplicate_branch_codes}")

print()
print("JOIN FAN-OUT DEMONSTRATION")
print("=" * 65)

print(f"Accounts before join:           {before_join:,}")
print(f"Rows after branch join:         {after_join:,}")
print(f"Extra rows created by fan-out:  {fanout_rows:,}")

print()
print("BALANCE BUSINESS-RULE TEST")
print("=" * 65)

print(f"Target account:                 {target_account}")
print(
    f">300% balance movements found:  "
    f"{len(balance_violations)}"
)

if len(balance_violations) > 0:
    print()
    print(
        balance_violations[
            [
                "account_id",
                "balance_date",
                "daily_balance",
                "pct_change"
            ]
        ].to_string(index=False)
    )

print()
print("✓ Bad dataset created separately from clean baseline")
print("✓ Join fan-out failure successfully reproduced")
print("✓ >300% balance movement injected")

DELIBERATE QUALITY VIOLATIONS
Bad accounts rows:              2,001
Duplicate account rows:         2
Null customer IDs:              1
Orphan branch codes:            1
Duplicate branch-key rows:      2

JOIN FAN-OUT DEMONSTRATION
Accounts before join:           2,001
Rows after branch join:         2,044
Extra rows created by fan-out:  43

BALANCE BUSINESS-RULE TEST
Target account:                 A000001
>300% balance movements found:  1

account_id balance_date  daily_balance  pct_change
   A000001   2026-08-22       19855.35         4.0

✓ Bad dataset created separately from clean baseline
✓ Join fan-out failure successfully reproduced
✓ >300% balance movement injected


In [ ]:
# CELL 4 — Create layered automated test suite

test_code = r'''
import os
from pathlib import Path

import pandas as pd
import pytest

PROJECT = Path(__file__).resolve().parents[1]

# CI and local runs can choose clean or bad data.
DATASET = os.getenv("LUMEN_DATASET", "clean")
DATA = PROJECT / "data" / DATASET


@pytest.fixture(scope="module")
def accounts():
    return pd.read_csv(
        DATA / "accounts.csv",
        parse_dates=["opened_date"]
    )


@pytest.fixture(scope="module")
def customers():
    return pd.read_csv(DATA / "customers.csv")


@pytest.fixture(scope="module")
def branches():
    return pd.read_csv(DATA / "branch_reference.csv")


@pytest.fixture(scope="module")
def balances():
    return pd.read_csv(
        DATA / "daily_balances.csv",
        parse_dates=["balance_date"]
    )


# --------------------------------------------------
# SCHEMA AND TYPE CHECKS
# --------------------------------------------------

def test_accounts_schema(accounts):
    expected_columns = [
        "account_id",
        "customer_id",
        "branch_code",
        "account_type",
        "status",
        "opened_date"
    ]

    assert list(accounts.columns) == expected_columns, (
        f"Accounts schema mismatch. "
        f"Expected {expected_columns}, "
        f"got {list(accounts.columns)}"
    )


def test_daily_balance_is_numeric(balances):
    assert pd.api.types.is_numeric_dtype(
        balances["daily_balance"]
    ), "daily_balance must be numeric"


# --------------------------------------------------
# PRIMARY KEY UNIQUENESS
# --------------------------------------------------

def test_account_id_is_unique(accounts):
    duplicates = accounts[
        accounts["account_id"].duplicated(keep=False)
    ]

    assert duplicates.empty, (
        "Duplicate account IDs found:\n"
        + duplicates.to_string(index=False)
    )


def test_branch_code_is_unique(branches):
    duplicates = branches[
        branches["branch_code"].duplicated(keep=False)
    ]

    assert duplicates.empty, (
        "Duplicate branch codes found. "
        "This can cause join fan-out:\n"
        + duplicates.to_string(index=False)
    )


# --------------------------------------------------
# NOT-NULL CHECKS
# --------------------------------------------------

def test_required_account_fields_not_null(accounts):
    required = [
        "account_id",
        "customer_id",
        "branch_code",
        "account_type",
        "status",
        "opened_date"
    ]

    failures = accounts[
        accounts[required].isna().any(axis=1)
    ]

    assert failures.empty, (
        "Required account fields contain nulls:\n"
        + failures.to_string(index=False)
    )


# --------------------------------------------------
# REFERENTIAL RELATIONSHIPS
# --------------------------------------------------

def test_accounts_reference_valid_customers(
    accounts,
    customers
):
    failures = accounts[
        ~accounts["customer_id"].isin(
            customers["customer_id"]
        )
    ]

    assert failures.empty, (
        "Accounts reference missing customers:\n"
        + failures.to_string(index=False)
    )


def test_accounts_reference_valid_branches(
    accounts,
    branches
):
    failures = accounts[
        ~accounts["branch_code"].isin(
            branches["branch_code"]
        )
    ]

    assert failures.empty, (
        "Accounts reference missing branches:\n"
        + failures.to_string(index=False)
    )


# --------------------------------------------------
# JOIN FAN-OUT TEST
# --------------------------------------------------

def test_branch_join_does_not_fan_out(
    accounts,
    branches
):
    before = len(accounts)

    joined = accounts.merge(
        branches,
        on="branch_code",
        how="left"
    )

    after = len(joined)

    assert after == before, (
        f"JOIN FAN-OUT DETECTED: "
        f"{before:,} account rows became "
        f"{after:,} rows after joining branch_reference. "
        f"Fan-out created {after - before:,} extra rows."
    )


# --------------------------------------------------
# BUSINESS RULE
# >300% DAY-OVER-DAY BALANCE MOVEMENT
# --------------------------------------------------

def test_balance_change_not_over_300_percent(
    balances
):
    ordered = balances.sort_values(
        ["account_id", "balance_date"]
    ).copy()

    ordered["previous_balance"] = (
        ordered.groupby("account_id")[
            "daily_balance"
        ].shift(1)
    )

    # Percent movement relative to previous day's balance.
    # Ignore a zero previous balance because percent change
    # is undefined in that situation.
    eligible = ordered[
        ordered["previous_balance"] > 0
    ].copy()

    eligible["pct_change"] = (
        (
            eligible["daily_balance"]
            - eligible["previous_balance"]
        ).abs()
        / eligible["previous_balance"]
    )

    failures = eligible[
        eligible["pct_change"] > 3.0
    ]

    assert failures.empty, (
        "Daily balance movement exceeded 300%:\n"
        + failures[
            [
                "account_id",
                "balance_date",
                "previous_balance",
                "daily_balance",
                "pct_change"
            ]
        ].to_string(index=False)
    )
'''

test_file = PROJECT / "tests" / "test_data_quality.py"
test_file.write_text(test_code)

requirements = """pandas
numpy
pytest
PyYAML
"""

(PROJECT / "requirements.txt").write_text(requirements)

print("LAYERED TEST SUITE CREATED")
print("=" * 60)
print("✓ Schema checks")
print("✓ Data-type checks")
print("✓ Account primary-key uniqueness")
print("✓ Branch primary-key uniqueness")
print("✓ Required-field not-null checks")
print("✓ Customer relationship checks")
print("✓ Branch relationship checks")
print("✓ Join fan-out protection")
print("✓ >300% daily-balance movement rule")
print()
print(f"Test file: {test_file}")

LAYERED TEST SUITE CREATED
✓ Schema checks
✓ Data-type checks
✓ Account primary-key uniqueness
✓ Branch primary-key uniqueness
✓ Required-field not-null checks
✓ Customer relationship checks
✓ Branch relationship checks
✓ Join fan-out protection
✓ >300% daily-balance movement rule

Test file: /content/lumen-community-bank-quality-gates/tests/test_data_quality.py


In [ ]:
# CELL 5 — Prove clean data passes and bad data is blocked

import subprocess
import os

def run_quality_tests(dataset):
    env = os.environ.copy()
    env["LUMEN_DATASET"] = dataset

    result = subprocess.run(
        [
            "python",
            "-m",
            "pytest",
            "tests/test_data_quality.py",
            "-v"
        ],
        cwd=PROJECT,
        env=env,
        capture_output=True,
        text=True
    )

    return result


# Run against clean data
print("=" * 70)
print("CLEAN DATASET")
print("=" * 70)

clean_result = run_quality_tests("clean")

print(clean_result.stdout)

if clean_result.stderr:
    print(clean_result.stderr)

print(f"CLEAN EXIT CODE: {clean_result.returncode}")


# Run against deliberately bad data
print("\n" + "=" * 70)
print("DELIBERATELY BAD DATASET")
print("=" * 70)

bad_result = run_quality_tests("bad")

print(bad_result.stdout)

if bad_result.stderr:
    print(bad_result.stderr)

print(f"BAD EXIT CODE: {bad_result.returncode}")


# Final gate decision
print("\n" + "=" * 70)
print("QUALITY GATE RESULT")
print("=" * 70)

if clean_result.returncode == 0:
    print("✓ CLEAN DATA PASSED")
else:
    print("✗ CLEAN DATA FAILED")

if bad_result.returncode != 0:
    print("✓ BAD DATA WAS BLOCKED")
else:
    print("✗ BAD DATA INCORRECTLY PASSED")

if clean_result.returncode == 0 and bad_result.returncode != 0:
    print()
    print("✓ QUALITY GATE BEHAVES CORRECTLY")
    print("✓ Good changes can proceed")
    print("✓ Bad changes are stopped before promotion")

CLEAN DATASET
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/lumen-community-bank-quality-gates
plugins: langsmith-0.14.2, typeguard-4.6.0, anyio-4.15.1, platformdirs-4.12.2
collecting ... collected 9 items

tests/test_data_quality.py::test_accounts_schema PASSED                  [ 11%]
tests/test_data_quality.py::test_daily_balance_is_numeric PASSED         [ 22%]
tests/test_data_quality.py::test_account_id_is_unique PASSED             [ 33%]
tests/test_data_quality.py::test_branch_code_is_unique PASSED            [ 44%]
tests/test_data_quality.py::test_required_account_fields_not_null PASSED [ 55%]
tests/test_data_quality.py::test_accounts_reference_valid_customers PASSED [ 66%]
tests/test_data_quality.py::test_accounts_reference_valid_branches PASSED [ 77%]
tests/test_data_quality.py::test_branch_join_does_not_fan_out PASSED    

In [ ]:
# CELL 6 — Create producer-consumer data contract

contract_yaml = """
contract:
  name: lumen_accounts_contract
  version: "1.0.0"
  dataset: accounts
  owner: Lumen Data Engineering
  consumer: Deposits Analytics

  freshness:
    max_age_hours: 24

  schema:
    account_id:
      type: string
      nullable: false
      unique: true

    customer_id:
      type: string
      nullable: false

    branch_code:
      type: string
      nullable: false

    account_type:
      type: string
      nullable: false
      allowed_values:
        - CHECKING
        - SAVINGS
        - MONEY_MARKET

    status:
      type: string
      nullable: false
      allowed_values:
        - ACTIVE
        - CLOSED
        - FROZEN

    opened_date:
      type: date
      nullable: false
      min_value: "2000-01-01"
      max_value: "2026-12-31"

  breaking_change_policy:
    notification_days: 14
    requires_consumer_approval: true
    breaking_changes:
      - removing_required_column
      - renaming_column
      - changing_column_type
      - making_required_field_nullable
      - adding_unapproved_enum_value

  policy:
    validation_stage: ingestion
    violation_action: reject
"""

contract_path = (
    PROJECT
    / "contracts"
    / "accounts_contract.yaml"
)

contract_path.write_text(contract_yaml.strip() + "\n")

print("PRODUCER-CONSUMER CONTRACT CREATED")
print("=" * 65)
print(f"Contract: {contract_path}")
print()
print(contract_path.read_text())

PRODUCER-CONSUMER CONTRACT CREATED
Contract: /content/lumen-community-bank-quality-gates/contracts/accounts_contract.yaml

contract:
  name: lumen_accounts_contract
  version: "1.0.0"
  dataset: accounts
  owner: Lumen Data Engineering
  consumer: Deposits Analytics

  freshness:
    max_age_hours: 24

  schema:
    account_id:
      type: string
      nullable: false
      unique: true

    customer_id:
      type: string
      nullable: false

    branch_code:
      type: string
      nullable: false

    account_type:
      type: string
      nullable: false
      allowed_values:
        - CHECKING
        - SAVINGS
        - MONEY_MARKET

    status:
      type: string
      nullable: false
      allowed_values:
        - ACTIVE
        - CLOSED
        - FROZEN

    opened_date:
      type: date
      nullable: false
      min_value: "2000-01-01"
      max_value: "2026-12-31"

  breaking_change_policy:
    notification_days: 14
    requires_consumer_approval: true
    breaking_cha

In [ ]:
# CELL 7 — Create contract validator

validator_code = r'''
import sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import yaml


def fail(message, violations):
    violations.append(message)
    print(f"VIOLATION: {message}")


def validate(contract_path, data_path):
    with open(contract_path, "r") as f:
        config = yaml.safe_load(f)["contract"]

    df = pd.read_csv(data_path)

    violations = []

    print("=" * 65)
    print("LUMEN ACCOUNTS CONTRACT VALIDATION")
    print("=" * 65)
    print(f"Contract: {config['name']}")
    print(f"Version:  {config['version']}")
    print(f"Dataset:  {data_path}")
    print()

    schema = config["schema"]

    # --------------------------------------------------
    # REQUIRED COLUMNS
    # --------------------------------------------------

    for column in schema:
        if column not in df.columns:
            fail(
                f"Required column '{column}' is missing.",
                violations
            )

    # Stop column-specific checks if columns are missing
    existing_columns = [
        col for col in schema if col in df.columns
    ]

    # --------------------------------------------------
    # NULLABILITY
    # --------------------------------------------------

    for column in existing_columns:
        rules = schema[column]

        if rules.get("nullable") is False:
            null_count = df[column].isna().sum()

            if null_count > 0:
                fail(
                    f"{column} contains "
                    f"{null_count} null value(s).",
                    violations
                )

    # --------------------------------------------------
    # UNIQUENESS
    # --------------------------------------------------

    for column in existing_columns:
        rules = schema[column]

        if rules.get("unique") is True:
            duplicate_count = (
                df[column]
                .duplicated(keep=False)
                .sum()
            )

            if duplicate_count > 0:
                fail(
                    f"{column} contains "
                    f"{duplicate_count} duplicate row(s).",
                    violations
                )

    # --------------------------------------------------
    # STRING TYPE CHECKS
    # --------------------------------------------------

    for column in existing_columns:
        rules = schema[column]

        if rules.get("type") == "string":
            non_null = df[column].dropna()

            invalid = non_null[
                ~non_null.map(
                    lambda value: isinstance(value, str)
                )
            ]

            if not invalid.empty:
                fail(
                    f"{column} contains non-string values.",
                    violations
                )

    # --------------------------------------------------
    # ALLOWED VALUES
    # --------------------------------------------------

    for column in existing_columns:
        rules = schema[column]

        allowed = rules.get("allowed_values")

        if allowed:
            invalid_values = sorted(
                df.loc[
                    df[column].notna()
                    & ~df[column].isin(allowed),
                    column
                ]
                .astype(str)
                .unique()
                .tolist()
            )

            if invalid_values:
                fail(
                    f"{column} contains forbidden value(s): "
                    f"{invalid_values}. "
                    f"Allowed values: {allowed}",
                    violations
                )

    # --------------------------------------------------
    # DATE RANGE
    # --------------------------------------------------

    if "opened_date" in df.columns:
        dates = pd.to_datetime(
            df["opened_date"],
            errors="coerce"
        )

        invalid_dates = (
            df["opened_date"].notna()
            & dates.isna()
        )

        if invalid_dates.any():
            fail(
                "opened_date contains invalid date values.",
                violations
            )

        date_rules = schema["opened_date"]

        minimum = pd.Timestamp(
            date_rules["min_value"]
        )

        maximum = pd.Timestamp(
            date_rules["max_value"]
        )

        out_of_range = (
            dates.notna()
            & (
                (dates < minimum)
                | (dates > maximum)
            )
        )

        if out_of_range.any():
            fail(
                f"opened_date contains "
                f"{out_of_range.sum()} value(s) "
                f"outside {minimum.date()} through "
                f"{maximum.date()}.",
                violations
            )

    # --------------------------------------------------
    # FRESHNESS SLA
    # --------------------------------------------------
    # For this synthetic project, freshness is checked
    # against the file modification time.

    max_age_hours = config["freshness"][
        "max_age_hours"
    ]

    file_time = datetime.fromtimestamp(
        Path(data_path).stat().st_mtime,
        tz=timezone.utc
    )

    now = datetime.now(timezone.utc)

    age_hours = (
        now - file_time
    ).total_seconds() / 3600

    if age_hours > max_age_hours:
        fail(
            f"Dataset is {age_hours:.1f} hours old. "
            f"SLA requires <= {max_age_hours} hours.",
            violations
        )

    # --------------------------------------------------
    # RESULT
    # --------------------------------------------------

    print()
    print("=" * 65)

    if violations:
        print(
            f"CONTRACT FAILED: "
            f"{len(violations)} violation(s)"
        )
        print("Promotion blocked.")
        return 1

    print("CONTRACT PASSED")
    print("All producer commitments satisfied.")
    print("Dataset may proceed to the next gate.")

    return 0


if __name__ == "__main__":

    if len(sys.argv) != 3:
        print(
            "Usage: python validate_contract.py "
            "<contract.yaml> <accounts.csv>"
        )
        sys.exit(2)

    exit_code = validate(
        sys.argv[1],
        sys.argv[2]
    )

    sys.exit(exit_code)
'''

validator_path = (
    PROJECT
    / "src"
    / "validate_contract.py"
)

validator_path.write_text(validator_code)

print("CONTRACT VALIDATOR CREATED")
print("=" * 65)
print(f"Validator: {validator_path}")
print()
print("✓ Schema validation")
print("✓ Required-column validation")
print("✓ Nullability validation")
print("✓ Uniqueness validation")
print("✓ Allowed-value validation")
print("✓ Date-range validation")
print("✓ Freshness SLA validation")
print("✓ Nonzero exit code on violation")

CONTRACT VALIDATOR CREATED
Validator: /content/lumen-community-bank-quality-gates/src/validate_contract.py

✓ Schema validation
✓ Required-column validation
✓ Nullability validation
✓ Uniqueness validation
✓ Allowed-value validation
✓ Date-range validation
✓ Freshness SLA validation
✓ Nonzero exit code on violation


In [ ]:
# CELL 8 — Demonstrate contract pass and contract failure

import subprocess
import pandas as pd

CONTRACT = PROJECT / "contracts" / "accounts_contract.yaml"
VALIDATOR = PROJECT / "src" / "validate_contract.py"

clean_accounts_path = (
    PROJECT / "data" / "clean" / "accounts.csv"
)

broken_accounts_path = (
    PROJECT / "data" / "contract_violation_accounts.csv"
)

# --------------------------------------------------
# CREATE DELIBERATE CONTRACT VIOLATION
# --------------------------------------------------

broken_accounts = pd.read_csv(clean_accounts_path)

# Introduce an unapproved enum value.
# Contract only allows ACTIVE, CLOSED, and FROZEN.
broken_accounts.loc[0, "status"] = "SUSPENDED"

broken_accounts.to_csv(
    broken_accounts_path,
    index=False
)

print("DELIBERATE BREAKING CHANGE")
print("=" * 65)
print("Changed account:", broken_accounts.loc[0, "account_id"])
print("Changed status: ", broken_accounts.loc[0, "status"])
print("Contract allows: ACTIVE, CLOSED, FROZEN")
print()


def run_contract_validation(data_path):
    result = subprocess.run(
        [
            "python",
            str(VALIDATOR),
            str(CONTRACT),
            str(data_path)
        ],
        cwd=PROJECT,
        capture_output=True,
        text=True
    )

    return result


# --------------------------------------------------
# CLEAN DATA
# --------------------------------------------------

print("=" * 65)
print("TEST 1 — CLEAN ACCOUNTS")
print("=" * 65)

clean_contract_result = run_contract_validation(
    clean_accounts_path
)

print(clean_contract_result.stdout)

if clean_contract_result.stderr:
    print(clean_contract_result.stderr)

print(
    "EXIT CODE:",
    clean_contract_result.returncode
)


# --------------------------------------------------
# BROKEN DATA
# --------------------------------------------------

print("\n" + "=" * 65)
print("TEST 2 — CONTRACT-BREAKING ACCOUNTS")
print("=" * 65)

broken_contract_result = run_contract_validation(
    broken_accounts_path
)

print(broken_contract_result.stdout)

if broken_contract_result.stderr:
    print(broken_contract_result.stderr)

print(
    "EXIT CODE:",
    broken_contract_result.returncode
)


# --------------------------------------------------
# FINAL RESULT
# --------------------------------------------------

print("\n" + "=" * 65)
print("CONTRACT GATE RESULT")
print("=" * 65)

if clean_contract_result.returncode == 0:
    print("✓ CLEAN ACCOUNTS PASSED")
else:
    print("✗ CLEAN ACCOUNTS FAILED")

if broken_contract_result.returncode != 0:
    print("✓ BREAKING CHANGE WAS BLOCKED")
else:
    print("✗ BREAKING CHANGE INCORRECTLY PASSED")

if (
    clean_contract_result.returncode == 0
    and broken_contract_result.returncode != 0
):
    print()
    print("✓ PRODUCER-CONSUMER CONTRACT IS ENFORCED")
    print("✓ Unapproved SUSPENDED status cannot be promoted")

DELIBERATE BREAKING CHANGE
Changed account: A000001
Changed status:  SUSPENDED
Contract allows: ACTIVE, CLOSED, FROZEN

TEST 1 — CLEAN ACCOUNTS
LUMEN ACCOUNTS CONTRACT VALIDATION
Contract: lumen_accounts_contract
Version:  1.0.0
Dataset:  /content/lumen-community-bank-quality-gates/data/clean/accounts.csv


CONTRACT PASSED
All producer commitments satisfied.
Dataset may proceed to the next gate.

EXIT CODE: 0

TEST 2 — CONTRACT-BREAKING ACCOUNTS
LUMEN ACCOUNTS CONTRACT VALIDATION
Contract: lumen_accounts_contract
Version:  1.0.0
Dataset:  /content/lumen-community-bank-quality-gates/data/contract_violation_accounts.csv

VIOLATION: status contains forbidden value(s): ['SUSPENDED']. Allowed values: ['ACTIVE', 'CLOSED', 'FROZEN']

CONTRACT FAILED: 1 violation(s)
Promotion blocked.

EXIT CODE: 1

CONTRACT GATE RESULT
✓ CLEAN ACCOUNTS PASSED
✓ BREAKING CHANGE WAS BLOCKED

✓ PRODUCER-CONSUMER CONTRACT IS ENFORCED
✓ Unapproved SUSPENDED status cannot be promoted


In [ ]:
# CELL 9 — Create GitHub Actions CI/CD workflow

workflow = """name: Lumen Data Quality Gate

on:
  pull_request:
    branches:
      - main

  push:
    branches:
      - main

jobs:

  quality-gate:
    name: Quality Gate
    runs-on: ubuntu-latest

    steps:

      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run layered data tests
        env:
          LUMEN_DATASET: clean
        run: |
          python -m pytest tests/test_data_quality.py -v

      - name: Validate producer-consumer contract
        run: |
          python src/validate_contract.py \
            contracts/accounts_contract.yaml \
            data/clean/accounts.csv

  promote-staging:
    name: Promote to Staging
    needs: quality-gate

    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    runs-on: ubuntu-latest

    steps:

      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Create staging artifact
        run: |
          mkdir -p staging
          cp data/clean/accounts.csv staging/accounts.csv
          cp contracts/accounts_contract.yaml staging/accounts_contract.yaml

          echo "commit=${GITHUB_SHA}" > staging/release.txt
          echo "environment=staging" >> staging/release.txt

      - name: Upload staging artifact
        uses: actions/upload-artifact@v4
        with:
          name: lumen-staging-${{ github.sha }}
          path: staging/
"""

workflow_path = (
    PROJECT
    / ".github"
    / "workflows"
    / "quality_gate.yml"
)

workflow_path.write_text(workflow)

print("GITHUB ACTIONS WORKFLOW CREATED")
print("=" * 65)
print(f"Workflow: {workflow_path}")
print()
print("PULL REQUEST BEHAVIOR")
print("✓ Checkout repository")
print("✓ Install Python and dependencies")
print("✓ Run all layered pytest checks")
print("✓ Run producer-consumer contract validator")
print("✓ Any nonzero exit code fails the workflow")
print("✓ Failed status check can block merge")
print()
print("MAIN BRANCH BEHAVIOR")
print("✓ Quality gate runs again after merge")
print("✓ Successful main build promotes to staging")
print("✓ Staging artifact records the Git commit")

GITHUB ACTIONS WORKFLOW CREATED
Workflow: /content/lumen-community-bank-quality-gates/.github/workflows/quality_gate.yml

PULL REQUEST BEHAVIOR
✓ Checkout repository
✓ Install Python and dependencies
✓ Run all layered pytest checks
✓ Run producer-consumer contract validator
✓ Any nonzero exit code fails the workflow
✓ Failed status check can block merge

MAIN BRANCH BEHAVIOR
✓ Quality gate runs again after merge
✓ Successful main build promotes to staging
✓ Staging artifact records the Git commit


In [ ]:
# CELL 10 — Create README and rollback runbook

readme = """# Lumen Community Bank Quality Gates

## Project Overview

This repository demonstrates automated data-quality controls for
Lumen Community Bank's deposits-reporting pipeline.

The project was designed in response to a production incident where
a silent join fan-out against the branch reference table overstated
customer balances in the CFO's weekly deposits packet.

The goal is to prevent the same failure class from reaching
production again.

## Architecture

Producer Data
    |
    v
Producer-Consumer Contract
    |
    v
Contract Validation
    |
    v
Layered Data Tests
    |
    v
GitHub Actions Quality Gate
    |
    v
Staging Promotion
    |
    v
Production

A failed test or contract validation returns a nonzero exit code.
GitHub Actions therefore marks the quality gate as failed and the
pull request should not be merged.

## Synthetic Core-Banking Data

The project generates:

* 42 branch records
* 1,800 customers
* 2,000 accounts
* 90 days of balances
* 180,000 daily balance records

The clean baseline contains no known integrity violations.

A separate deliberately corrupted dataset demonstrates:

* duplicate account IDs
* null customer IDs
* orphan branch codes
* duplicate branch keys
* join fan-out
* greater than 300 percent daily balance movement

## Layered Test Suite

The pytest suite validates:

* schema
* data types
* account primary-key uniqueness
* branch primary-key uniqueness
* required-field nullability
* customer relationships
* branch relationships
* join row-count stability
* daily balance business rules

### Join Fan-Out Protection

The most important regression test compares account row counts before
and after joining to the branch reference table.

The deliberate bad dataset contains 2,001 account rows.

Joining to a branch reference table containing a duplicate B001 key
creates 2,044 rows.

The test detects the 43-row increase and fails the quality gate.

This distinguishes the surface symptom from the root cause.

Surface symptom:

The CFO's deposits report contains overstated balances.

Underlying root cause:

The branch reference table violates key uniqueness, causing accounts
to match multiple reference rows during the join.

## Producer-Consumer Contract

The accounts contract is stored at:

contracts/accounts_contract.yaml

It defines:

* schema
* data types
* nullability
* account ID uniqueness
* allowed account types
* allowed status values
* opened-date range
* 24-hour freshness SLA
* producing and consuming teams
* breaking-change policy
* 14-day notification period
* required consumer approval

The validator returns a nonzero exit code when the contract is
violated.

## Deliberate Contract Failure

The contract permits these account status values:

ACTIVE
CLOSED
FROZEN

A deliberate breaking change introduces:

SUSPENDED

The contract validator rejects this value and exits with code 1.

Surface symptom:

The GitHub Actions contract-validation check fails.

Underlying root cause:

The producer introduced a new business value without following the
breaking-change process or obtaining consumer approval.

The correct response is not simply to weaken the contract. The team
must determine whether the new value is legitimate and coordinate the
contract change with downstream consumers.

## CI/CD Promotion Path

Every pull request to main runs:

1. dependency installation
2. layered pytest suite
3. producer-consumer contract validation

Any failure stops the quality-gate job.

After a successful merge to main, GitHub Actions runs the quality
gate again and creates a staging artifact containing the validated
accounts dataset, contract, and source Git commit.

## Branch Protection

The main branch should require the Quality Gate status check before
merge.

This prevents contributors from bypassing failed automated controls.

## Rollback Runbook

### When to Roll Back

Rollback should be considered when a merged change causes incorrect
data, breaks a downstream consumer, introduces an unexpected schema
change, or produces incorrect deposits-reporting results.

### Code Rollback

Identify the bad merge commit:

git log --oneline

Create a revert:

git revert <bad_commit_sha>

Push the revert through the normal review and quality-gate process.

Expected recovery time:

Approximately 5 to 15 minutes after approval and successful CI.

Use this option when the defect is caused by code, configuration,
contract logic, or transformation behavior.

### Dataset Rollback

Each staging artifact is associated with the Git commit that produced
it.

Locate the most recent known-good staging artifact in GitHub Actions
and restore the validated dataset associated with that commit.

Expected recovery time:

Approximately 10 to 30 minutes depending on artifact size and
validation time.

Use this option when published data is incorrect but the application
code does not need to change.

### How to Choose

Use a code revert when the root cause is code or configuration.

Use a dataset restore when the pipeline code is correct but an
incorrect dataset version was published.

If both code and data are affected, revert the code first and then
restore or rebuild the dataset from the last known-good version.

After rollback, rerun the full quality gate before republishing data.

## Repository Structure

src/
    validate_contract.py

tests/
    test_data_quality.py

contracts/
    accounts_contract.yaml

data/
    clean/
    bad/

.github/
    workflows/
        quality_gate.yml

staging/

README.md
requirements.txt
"""

readme_path = PROJECT / "README.md"
readme_path.write_text(readme)

print("README + ROLLBACK RUNBOOK CREATED")
print("=" * 65)
print(f"README: {readme_path}")
print()
print("✓ Project purpose documented")
print("✓ Join fan-out incident documented")
print("✓ Surface symptom vs root cause documented")
print("✓ Contract enforcement documented")
print("✓ CI/CD promotion path documented")
print("✓ Code rollback documented")
print("✓ Dataset rollback documented")
print("✓ Recovery-time expectations documented")

README + ROLLBACK RUNBOOK CREATED
README: /content/lumen-community-bank-quality-gates/README.md

✓ Project purpose documented
✓ Join fan-out incident documented
✓ Surface symptom vs root cause documented
✓ Contract enforcement documented
✓ CI/CD promotion path documented
✓ Code rollback documented
✓ Dataset rollback documented
✓ Recovery-time expectations documented


In [ ]:
# CELL 11 — Final local repository audit

import subprocess
from pathlib import Path
import os

required_files = [
    "README.md",
    "requirements.txt",
    "src/validate_contract.py",
    "tests/test_data_quality.py",
    "contracts/accounts_contract.yaml",
    ".github/workflows/quality_gate.yml",
    "data/clean/accounts.csv",
    "data/clean/customers.csv",
    "data/clean/branch_reference.csv",
    "data/clean/daily_balances.csv",
    "data/bad/accounts.csv",
    "data/bad/customers.csv",
    "data/bad/branch_reference.csv",
    "data/bad/daily_balances.csv",
]

print("=" * 70)
print("FINAL REPOSITORY AUDIT")
print("=" * 70)

missing = []

for file in required_files:
    path = PROJECT / file

    if path.exists():
        print(f"✓ {file}")
    else:
        print(f"✗ MISSING: {file}")
        missing.append(file)

print()
print("=" * 70)
print("CLEAN PYTEST GATE")
print("=" * 70)

env = os.environ.copy()
env["LUMEN_DATASET"] = "clean"

pytest_result = subprocess.run(
    [
        "python",
        "-m",
        "pytest",
        "tests/test_data_quality.py",
        "-q"
    ],
    cwd=PROJECT,
    env=env,
    capture_output=True,
    text=True
)

print(pytest_result.stdout)
print("PYTEST EXIT CODE:", pytest_result.returncode)

print()
print("=" * 70)
print("CLEAN CONTRACT GATE")
print("=" * 70)

contract_result = subprocess.run(
    [
        "python",
        "src/validate_contract.py",
        "contracts/accounts_contract.yaml",
        "data/clean/accounts.csv"
    ],
    cwd=PROJECT,
    capture_output=True,
    text=True
)

print(contract_result.stdout)
print("CONTRACT EXIT CODE:", contract_result.returncode)

print()
print("=" * 70)
print("FINAL STATUS")
print("=" * 70)

if (
    not missing
    and pytest_result.returncode == 0
    and contract_result.returncode == 0
):
    print("✓ ALL REQUIRED FILES PRESENT")
    print("✓ ALL 9 CLEAN DATA TESTS PASS")
    print("✓ PRODUCER-CONSUMER CONTRACT PASSES")
    print("✓ LOCAL QUALITY GATE PASSES")
    print("✓ REPOSITORY READY FOR GITHUB")
else:
    print("✗ REPOSITORY IS NOT READY YET")

    if missing:
        print(f"Missing files: {missing}")

FINAL REPOSITORY AUDIT
✓ README.md
✓ requirements.txt
✓ src/validate_contract.py
✓ tests/test_data_quality.py
✓ contracts/accounts_contract.yaml
✓ .github/workflows/quality_gate.yml
✓ data/clean/accounts.csv
✓ data/clean/customers.csv
✓ data/clean/branch_reference.csv
✓ data/clean/daily_balances.csv
✓ data/bad/accounts.csv
✓ data/bad/customers.csv
✓ data/bad/branch_reference.csv
✓ data/bad/daily_balances.csv

CLEAN PYTEST GATE
.........                                                                [100%]
9 passed in 0.69s

PYTEST EXIT CODE: 0

CLEAN CONTRACT GATE
LUMEN ACCOUNTS CONTRACT VALIDATION
Contract: lumen_accounts_contract
Version:  1.0.0
Dataset:  data/clean/accounts.csv


CONTRACT PASSED
All producer commitments satisfied.
Dataset may proceed to the next gate.

CONTRACT EXIT CODE: 0

FINAL STATUS
✓ ALL REQUIRED FILES PRESENT
✓ ALL 9 CLEAN DATA TESTS PASS
✓ PRODUCER-CONSUMER CONTRACT PASSES
✓ LOCAL QUALITY GATE PASSES
✓ REPOSITORY READY FOR GITHUB


In [ ]:
# CELL 11 — Final local repository audit

import subprocess
from pathlib import Path
import os

required_files = [
    "README.md",
    "requirements.txt",
    "src/validate_contract.py",
    "tests/test_data_quality.py",
    "contracts/accounts_contract.yaml",
    ".github/workflows/quality_gate.yml",
    "data/clean/accounts.csv",
    "data/clean/customers.csv",
    "data/clean/branch_reference.csv",
    "data/clean/daily_balances.csv",
    "data/bad/accounts.csv",
    "data/bad/customers.csv",
    "data/bad/branch_reference.csv",
    "data/bad/daily_balances.csv",
]

print("=" * 70)
print("FINAL REPOSITORY AUDIT")
print("=" * 70)

missing = []

for file in required_files:
    path = PROJECT / file

    if path.exists():
        print(f"✓ {file}")
    else:
        print(f"✗ MISSING: {file}")
        missing.append(file)

print()
print("=" * 70)
print("CLEAN PYTEST GATE")
print("=" * 70)

env = os.environ.copy()
env["LUMEN_DATASET"] = "clean"

pytest_result = subprocess.run(
    [
        "python",
        "-m",
        "pytest",
        "tests/test_data_quality.py",
        "-q"
    ],
    cwd=PROJECT,
    env=env,
    capture_output=True,
    text=True
)

print(pytest_result.stdout)
print("PYTEST EXIT CODE:", pytest_result.returncode)

print()
print("=" * 70)
print("CLEAN CONTRACT GATE")
print("=" * 70)

contract_result = subprocess.run(
    [
        "python",
        "src/validate_contract.py",
        "contracts/accounts_contract.yaml",
        "data/clean/accounts.csv"
    ],
    cwd=PROJECT,
    capture_output=True,
    text=True
)

print(contract_result.stdout)
print("CONTRACT EXIT CODE:", contract_result.returncode)

print()
print("=" * 70)
print("FINAL STATUS")
print("=" * 70)

if (
    not missing
    and pytest_result.returncode == 0
    and contract_result.returncode == 0
):
    print("✓ ALL REQUIRED FILES PRESENT")
    print("✓ ALL 9 CLEAN DATA TESTS PASS")
    print("✓ PRODUCER-CONSUMER CONTRACT PASSES")
    print("✓ LOCAL QUALITY GATE PASSES")
    print("✓ REPOSITORY READY FOR GITHUB")
else:
    print("✗ REPOSITORY IS NOT READY YET")

    if missing:
        print(f"Missing files: {missing}")

FINAL REPOSITORY AUDIT
✓ README.md
✓ requirements.txt
✓ src/validate_contract.py
✓ tests/test_data_quality.py
✓ contracts/accounts_contract.yaml
✓ .github/workflows/quality_gate.yml
✓ data/clean/accounts.csv
✓ data/clean/customers.csv
✓ data/clean/branch_reference.csv
✓ data/clean/daily_balances.csv
✓ data/bad/accounts.csv
✓ data/bad/customers.csv
✓ data/bad/branch_reference.csv
✓ data/bad/daily_balances.csv

CLEAN PYTEST GATE
.........                                                                [100%]
9 passed in 0.81s

PYTEST EXIT CODE: 0

CLEAN CONTRACT GATE
LUMEN ACCOUNTS CONTRACT VALIDATION
Contract: lumen_accounts_contract
Version:  1.0.0
Dataset:  data/clean/accounts.csv


CONTRACT PASSED
All producer commitments satisfied.
Dataset may proceed to the next gate.

CONTRACT EXIT CODE: 0

FINAL STATUS
✓ ALL REQUIRED FILES PRESENT
✓ ALL 9 CLEAN DATA TESTS PASS
✓ PRODUCER-CONSUMER CONTRACT PASSES
✓ LOCAL QUALITY GATE PASSES
✓ REPOSITORY READY FOR GITHUB


In [ ]:
# CELL 12 — Initialize Git repository and create baseline commit

import subprocess
from pathlib import Path

def git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=PROJECT,
        capture_output=True,
        text=True
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    return result


# Create .gitignore
gitignore = """
__pycache__/
.pytest_cache/
*.pyc
.ipynb_checkpoints/
.DS_Store
""".strip() + "\n"

(PROJECT / ".gitignore").write_text(gitignore)


# Initialize repository
git("init")

# Use main as the primary branch
git("branch", "-M", "main")

# Git identity
git("config", "user.name", "Jared Everson")
git("config", "user.email", "eversonj1@csp.edu")

# Add project files
git("add", ".")

# Initial commit
git(
    "commit",
    "-m",
    "Build Lumen data quality gates and CI pipeline"
)

print("=" * 70)
print("GIT STATUS")
print("=" * 70)
git("status")

print("=" * 70)
print("GIT HISTORY")
print("=" * 70)
git("--no-pager", "log", "--oneline", "-5")

Initialized empty Git repository in /content/lumen-community-bank-quality-gates/.git/

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>

[main (root-commit) 1c1885b] Build Lumen data quality gates and CI pipeline
 16 files changed, 370559 insertions(+)
 create mode 100644 .github/workflows/quality_gate.yml
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 contracts/accounts_contract.yaml
 create mode 100644 data/bad/accounts.csv
 create mode 100644 data/bad/branch_reference.csv
 create mode 100644 data/bad/customers.csv
 create

CompletedProcess(args=['git', '--no-pager', 'log', '--oneline', '-5'], returncode=0, stdout='1c1885b Build Lumen data quality gates and CI pipeline\n', stderr='')